# Лабораторная работа №3

**Вариант 3.2**

**Выполнили:** Аверьянова Мария, Калягин Дмитрий, Кашникова Анна, Климович Анна

## Функция оптимизации

Рассмотрим задачу минимизации:

$$\begin{aligned}
\text{minimize} \quad & f(x) = \sum_{i=1}^n x_i \log x_i \\
\text{subject to} \quad & Ax = b \\
& x \geq \mathbf{1}
\end{aligned}$$

где $A \in \mathbb{R}^{p \times n}$, $b \in \mathbb{R}^p$, $p < n$.

---

## 1. Логарифмический барьер и исследование на выпуклость

### 1.1. Прямая задача с логарифмическим барьером

Неравенства: $g_i(x) = 1 - x_i \leq 0$ (так как $x_i \geq 1$)

Логарифмический барьер:
$$B_L(x) = -\sum_{i=1}^n \log(x_i - 1)$$

Барьерная задача для фиксированного $t > 0$:
$$\min_x \left\{ t\sum_{i=1}^n x_i \log x_i - \sum_{i=1}^n \log(x_i - 1) \right\}$$
$$\text{subject to } Ax = b$$

**Исследование на выпуклость:**

Функция $f(x) = \sum_{i=1}^n x_i \log x_i$ выпукла, так как:
- $\frac{d^2}{dx_i^2}(x_i \log x_i) = \frac{1}{x_i} > 0$ при $x_i > 0$

Барьерная функция $-\log(x_i - 1)$ выпукла при $x_i > 1$, так как:
- $\frac{d^2}{dx_i^2}[-\log(x_i - 1)] = \frac{1}{(x_i-1)^2} > 0$

**Вывод:** Барьерная задача выпукла (сумма выпуклых функций).

---

### 1.2. Двойственная задача с логарифмическим барьером

Запишем ограничения как $1-x_i\le 0$ и введём множители $\lambda_i\ge 0$:

$$L(x,\nu,\lambda)=\sum_{i=1}^n x_i\log x_i+\nu^T(Ax-b)+\lambda^T(\mathbf 1-x).$$

Условие минимума по $x$:

$$\log x_i+1+a_i^T\nu-\lambda_i=0,$$

откуда

$$x_i(\nu,\lambda)=\exp(-1-a_i^T\nu+\lambda_i).$$

Двойственная функция:

$$q(\nu,\lambda)=-b^T\nu+\mathbf 1^T\lambda-\sum_{i=1}^n \exp(-1-a_i^T\nu+\lambda_i).$$

Двойственная задача максимизации:

$$\max_{\nu,\lambda}\;q(\nu,\lambda),\qquad \lambda\ge 0.$$

Эквивалентная задача минимизации:

$$\min_{\nu,\lambda}\;\varphi(\nu,\lambda)=b^T\nu-\mathbf 1^T\lambda+\sum_{i=1}^n \exp(-1-a_i^T\nu+\lambda_i),\qquad \lambda\ge 0.$$

Барьерная задача для двойственной задачи:

$$\min_{\nu,\lambda}\;t\varphi(\nu,\lambda)-\sum_{i=1}^n\log\lambda_i,\qquad \lambda_i>0.$$

**Исследование на выпуклость:**

Функция $\varphi(\nu,\lambda)$ выпукла, потому что состоит из линейной части и суммы экспонент от аффинных функций. Барьер $-\sum_i\log\lambda_i$ выпуклый на $\lambda_i>0$. Поэтому двойственная барьерная задача выпукла.


## 2. Генерация тестовых примеров и эталонное решение через CVXPY

In [1]:
import time
import pickle
import matplotlib.pyplot as plt
import numpy as np
import cvxpy as cp
import pandas as pd
from pathlib import Path
from scipy.linalg import svd
from scipy.optimize import minimize
from typing import List, Dict, Tuple
from scipy.sparse.linalg import cg

In [2]:
def entropy_objective(x: np.ndarray) -> float:
    if np.any(x <= 0):
        return np.inf
    return np.sum(x * np.log(x))

def recover_primal_from_dual(nu: np.ndarray, lam: np.ndarray, A: np.ndarray) -> np.ndarray:
    # return np.exp(-A.T @ nu - 1)
    return np.exp(-1 - A.T @ nu + lam)

def generate_full_rank_matrix(p: int, n: int, rng) -> np.ndarray:
    while True:
        A = rng.standard_normal((p, n))
        if np.linalg.matrix_rank(A) == p:
            return A

def nullspace_basis(A: np.ndarray, tol=1e-12) -> np.ndarray:
    U, S, Vt = svd(A, full_matrices=True)
    rank = np.sum(S > tol * S[0])
    return Vt[rank:].T

ГЕНЕРАЦИЯ ЗАДАЧИ

In [3]:
def generate_test_problem(n: int, p: int, seed: int, margin: float = 0.5):
    rng = np.random.default_rng(seed)

    A = generate_full_rank_matrix(p, n, rng)

    # Генерация допустимой точки x >= 1
    x_feas = 1.0 + margin + np.abs(rng.standard_normal(n))  # x >= 1

    b = A @ x_feas
    N = nullspace_basis(A)

    return {
        'A': A,
        'b': b,
        'x_feas': x_feas,
        'N': N,
        'rng': rng,
        'n': n,
        'p': p
    }

СТАРТОВЫЕ ТОЧКИ

In [4]:
def generate_feasible_starts(x_ref, N, num_starts, rng, margin=1e-4):
    starts = []
    n_p = N.shape[1]

    if n_p == 0:
        return [x_ref.copy() for _ in range(num_starts)]

    for _ in range(num_starts):
        direction = N @ rng.standard_normal(n_p)

        if np.linalg.norm(direction) < 1e-14:
            starts.append(x_ref.copy())
            continue

        lower, upper = -np.inf, np.inf

        for xi, di in zip(x_ref, direction):
            if abs(di) < 1e-14:
                continue

            bound = (1.0 + margin - xi) / di

            if di > 0:
                lower = max(lower, bound)
            else:
                upper = min(upper, bound)

        if not np.isfinite(lower):
            lower = -1.0
        if not np.isfinite(upper):
            upper = 1.0

        if lower >= upper:
            starts.append(x_ref.copy())
            continue

        alpha = rng.uniform(0.5 * lower, 0.5 * upper)
        x0 = x_ref + alpha * direction

        if np.min(x0) <= 1.0:
            x0 = x_ref.copy()

        starts.append(x0)

    return starts

CVXPY РЕШЕНИЕ

In [5]:
def solve_primal_cvxpy(A, b):
    n = A.shape[1]
    x = cp.Variable(n)

    objective = cp.Minimize(cp.sum(-cp.entr(x)))
    constraints = [A @ x == b, x >= 1.0]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.SCS, eps=1e-8, max_iters=10000)

    if x.value is None:
        raise RuntimeError("Primal solve failed")

    x_star = np.asarray(x.value).ravel()
    f_star = entropy_objective(x_star)

    return x_star, f_star


def solve_dual_cvxpy(A, b):
    p, n = A.shape
    nu = cp.Variable(p)
    lam = cp.Variable(n, nonneg=True)

    phi_expr = (
        b @ nu
        - cp.sum(lam)
        + cp.sum(
            cp.exp(-1 - A.T @ nu + lam)
        )
    )

    prob = cp.Problem(cp.Minimize(phi_expr))

    prob.solve(
        solver=cp.SCS,
        eps=1e-8,
        max_iters=10000
    )

    if nu.value is None or lam.value is None:
        raise RuntimeError("Dual solve failed")

    nu_star = np.asarray(nu.value).ravel()
    lam_star = np.asarray(lam.value).ravel()

    phi_star = float(prob.value)
    q_star = -phi_star

    return nu_star, lam_star, q_star, phi_star

In [6]:
def build_dataset(
    n_values=range(10, 101, 10),
    num_instances=5,
    num_starts=5,
    base_seed=1000,
    save_path=None,
):
    problems: List[Dict] = []
    global_problem_id = 0


    for n in n_values:
        p = n // 2  # p < n

        print(f"\n>>> n = {n}, p = {p}")

        for inst_id in range(num_instances):
            seed = base_seed * n + inst_id

            prob = generate_test_problem(n, p, seed)
            A = prob["A"]
            b = prob["b"]
            N = prob["N"]
            rng = prob["rng"]

            # CVXPY reference solutions
            x_star, f_star = solve_primal_cvxpy(A, b)
            nu_star, lam_star, q_star, phi_star = solve_dual_cvxpy(A, b)

            x_dual = recover_primal_from_dual(nu_star, lam_star, A)
            duality_gap = abs(f_star - q_star)

            # primal feasible starts
            primal_starts = generate_feasible_starts(
                prob["x_feas"], N, num_starts, rng
            )

            # dual starts
            dual_starts = [
                rng.standard_normal(p) * 0.1
                for _ in range(num_starts)
            ]


            problems.append({
                "problem_id": global_problem_id,
                "n": n,
                "p": p,
                "instance_id": inst_id,
                "seed": seed,
                "A": A,
                "b": b,
                "N": N,
                "x_feas": prob["x_feas"],
                "x_star": x_star,
                "f_star": f_star,
                "nu_star": nu_star,
                "lam_star" : lam_star,
                "q_star": q_star,
                "phi_star": phi_star,
                "x_dual": x_dual,
                "duality_gap": duality_gap,
                "primal_starts": primal_starts,
                "dual_starts": dual_starts,
            })

            print(f"  Instance {inst_id}: f* = {f_star:.4f}, "
                  f"q* = {q_star:.4f}, gap = {duality_gap:.2e}")

            global_problem_id += 1

    if save_path is not None:
        with open(save_path, "wb") as f:
            pickle.dump(problems, f)
        print(f"\nDataset saved to {save_path}")

    return problems

In [7]:
problems = build_dataset(
    n_values=range(10, 101, 10),
    num_instances=5,
    num_starts=5,
    save_path="entropy_problems.pkl",
)


>>> n = 10, p = 5
  Instance 0: f* = 12.1291, q* = 12.1291, gap = 2.17e-08
  Instance 1: f* = 6.8180, q* = 6.8180, gap = 1.04e-08
  Instance 2: f* = 16.1214, q* = 16.1214, gap = 1.57e-08
  Instance 3: f* = 10.2027, q* = 10.2027, gap = 2.25e-08
  Instance 4: f* = 14.8322, q* = 14.8322, gap = 1.63e-08

>>> n = 20, p = 10
  Instance 0: f* = 20.6607, q* = 20.6607, gap = 6.22e-08
  Instance 1: f* = 23.2151, q* = 23.2151, gap = 1.14e-07
  Instance 2: f* = 36.1472, q* = 36.1472, gap = 8.79e-08
  Instance 3: f* = 19.8717, q* = 19.8717, gap = 1.41e-08
  Instance 4: f* = 21.0615, q* = 21.0615, gap = 3.29e-07

>>> n = 30, p = 15
  Instance 0: f* = 33.5016, q* = 33.5016, gap = 1.67e-07
  Instance 1: f* = 48.4108, q* = 48.4108, gap = 5.59e-08
  Instance 2: f* = 28.4016, q* = 28.4016, gap = 3.05e-07
  Instance 3: f* = 39.1883, q* = 39.1883, gap = 5.96e-09
  Instance 4: f* = 22.8000, q* = 22.8000, gap = 4.73e-07

>>> n = 40, p = 20
  Instance 0: f* = 49.2144, q* = 49.2144, gap = 6.84e-07
  Instance 

## 3. Барьерный метод

In [8]:
def line_search_scalar(
    f,
    x,
    p,
    gamma=1.0,
    min_gamma=1e-8,
    num=30,
):
    f_x = f(x)

    if not np.isfinite(f_x):
        return 0.0

    gammas = np.geomspace(gamma, min_gamma, num)

    best_gamma = 0.0
    best_value = f_x

    for c_gamma in gammas:
        value = f(x + c_gamma * p)

        if np.isfinite(value) and value < best_value:
            best_value = value
            best_gamma = c_gamma

    return best_gamma

### 3.1 Барьерная задача - метод Ньютона 

In [9]:
def barrier_function_primal(x: np.ndarray, t: float) -> float:
    if np.any(x <= 1):
        return np.inf

    entropy_term = np.sum(x * np.log(x))
    barrier_term = np.sum(np.log(x - 1))

    return t * entropy_term - barrier_term

def barrier_gradient_primal(x: np.ndarray, t: float) -> np.ndarray:
    grad = t * (np.log(x) + 1) - 1.0 / (x - 1)
    return grad

def barrier_hessian_primal(x: np.ndarray, t: float) -> np.ndarray:
    diag = t / x + 1.0 / (x - 1) ** 2
    return np.diag(diag)

def newton_step_equality_constrained(
    x: np.ndarray, t: float, A: np.ndarray, b: np.ndarray
):

    n = len(x)
    p = A.shape[0]

    grad = barrier_gradient_primal(x, t)
    H = barrier_hessian_primal(x, t)

    KKT = np.block([
        [H, A.T],
        [A, np.zeros((p, p))]
    ])

    rhs = -np.concatenate([grad, A @ x - b])

    try:
        sol = np.linalg.solve(KKT, rhs)
    except np.linalg.LinAlgError:
        sol = np.linalg.lstsq(KKT, rhs, rcond=None)[0]

    return sol[:n]


def barrier_method_newton(
    problem,
    x0,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
    inner_tol=1e-8,
    max_inner_iter=50,
):
    """
    Барьерный метод Boyd Algorithm 11.1 для прямой задачи.
    - остановка внешнего цикла по m/t <= eps;
    """
    A = problem['A']
    b = problem['b']
    f_star = problem.get('f_star', np.nan)
    n = problem['n']
    m = n

    x = x0.copy()
    t = t0

    if np.any(x <= 1) or np.linalg.norm(A @ x - b) > 1e-6:
        raise ValueError("x0 must satisfy Ax=b and x0>1 for the primal barrier method")

    history = {
        'outer_iter': [],
        'inner_iter': [],
        'objective': [],
        'error': [],
        'barrier_gap': [],
        't_values': [],
        'total_newton_steps': 0
    }

    total_newton_steps = 0

    for outer_iter in range(max_iter):
        inner_iter = 0

        for _ in range(max_inner_iter):
            dx = newton_step_equality_constrained(x, t, A, b)

            H = barrier_hessian_primal(x, t)
            decrement_sq = float(dx @ H @ dx)

            if decrement_sq / 2 <= inner_tol:
                break

            grad = barrier_gradient_primal(x, t)
            step = line_search_scalar(
                f=lambda x_: barrier_function_primal(x_, t),
                x=x,
                p=dx,
                gamma=1.0,
                min_gamma=1e-8,
                num=30
            )

            if step <= 0:
                break

            x = x + step * dx

            inner_iter += 1
            total_newton_steps += 1

        f_val = entropy_objective(x)
        error = abs(f_val - f_star) if np.isfinite(f_star) else np.nan
        barrier_gap = m / t

        history['outer_iter'].append(outer_iter)
        history['inner_iter'].append(inner_iter)
        history['objective'].append(f_val)
        history['error'].append(error)
        history['barrier_gap'].append(barrier_gap)
        history['t_values'].append(t)
        history['total_newton_steps'] = total_newton_steps

        if barrier_gap <= eps:
            break

        t *= mu

    return x, f_val, history

In [10]:
def dual_phi(
    z: np.ndarray,
    A: np.ndarray,
    b: np.ndarray,
    t: float
) -> float:
    p = A.shape[0]
    nu = z[:p]
    lam = z[p:]

    if np.any(lam <= 0):
        return np.inf

    exp_term = np.exp(-1 - A.T @ nu + lam)

    objective = (
        t * (
            b @ nu
            - np.sum(lam)
            + np.sum(exp_term)
        )
        - np.sum(np.log(lam))
    )

    return objective


def dual_phi_grad(
    z: np.ndarray,
    A: np.ndarray,
    b: np.ndarray,
    t: float
) -> np.ndarray:
    p = A.shape[0]
    nu = z[:p]
    lam = z[p:]

    if np.any(lam <= 0):
        return np.full_like(z, np.inf)

    exp_term = np.exp(-1 - A.T @ nu + lam)

    grad_nu = t * (b - A @ exp_term)
    grad_lam = t * (-1.0 + exp_term) - 1.0 / lam

    return np.concatenate([grad_nu, grad_lam])


def dual_phi_hess(
    z: np.ndarray,
    A: np.ndarray,
    t: float
) -> np.ndarray:
    p, n = A.shape
    nu = z[:p]
    lam = z[p:]

    exp_term = np.exp(-1 - A.T @ nu + lam)

    H_nu_nu = t * ((A * exp_term) @ A.T)
    H_lam_lam = np.diag(t * exp_term + 1.0 / lam**2)
    H_nu_lam = -t * (A * exp_term)

    H = np.block([
        [H_nu_nu, H_nu_lam],
        [H_nu_lam.T, H_lam_lam]
    ])

    return H



def barrier_method_dual_newton(
    problem: Dict,
    dual_start: np.ndarray,
    mu: float = 10.0,
    t0: float = 1.0,
    eps: float = 0.012,
    max_iter: int = 100,
    inner_tol: float = 1e-8,
    max_inner_iter: int = 50,
) -> Tuple[np.ndarray, float, Dict]:

    A = problem['A']
    b = problem['b']
    q_star = problem['q_star']
    p = problem['p']
    n = problem['n']
    m = n

    dual_start = np.asarray(dual_start).ravel()

    # Поддерживаем два формата:
    # только nu0 длины p, lambda стартует с единиц
    # z0 = [nu0, lambda0] длины p+n
    if dual_start.size == p:
        nu0 = dual_start.copy()
        lam0 = np.ones(n)
        z = np.concatenate([nu0, lam0])
    elif dual_start.size == p + n:
        z = dual_start.copy()
        if np.any(z[p:] <= 0):
            raise ValueError("dual_start must have lambda > 0")
    else:
        raise ValueError(f"dual_start must have length p={p} or p+n={p+n}")

    t = t0

    history = {
        'outer_iter': [],
        'inner_iter': [],
        'objective': [],
        'error': [],
        'barrier_gap': [],
        't_values': [],
        'total_newton_steps': 0
    }

    total_newton_steps = 0
    q_val = np.nan

    for outer_iter in range(max_iter):
        inner_iter = 0

        for _ in range(max_inner_iter):
            grad = dual_phi_grad(z, A, b, t)
            H = dual_phi_hess(z, A, t)

            H += 1e-10 * np.eye(H.shape[0])

            try:
                dz = -np.linalg.solve(H, grad)
            except np.linalg.LinAlgError:
                dz = -np.linalg.lstsq(H, grad, rcond=None)[0]

            decrement_sq = float(-grad @ dz)

            if decrement_sq / 2 <= inner_tol:
                break

            step = line_search_scalar(
                f=lambda z_: dual_phi(z_, A, b, t),
                x=z,
                p=dz,
                gamma=1.0,
                min_gamma=1e-8,
                num=30
            )

            if step <= 1e-12:
                break

            z = z + step * dz

            inner_iter += 1
            total_newton_steps += 1

        nu = z[:p]
        lam = z[p:]

        exp_term = np.exp(-1 - A.T @ nu + lam)

        phi_val = (
            b @ nu
            - np.sum(lam)
            + np.sum(exp_term)
        )

        q_val = -phi_val

        error = abs(q_val - q_star)
        barrier_gap = m / t

        history['outer_iter'].append(outer_iter)
        history['inner_iter'].append(inner_iter)
        history['objective'].append(q_val)
        history['error'].append(error)
        history['barrier_gap'].append(barrier_gap)
        history['t_values'].append(t)
        history['total_newton_steps'] = total_newton_steps

        if barrier_gap <= eps:
            break

        t *= mu

    return z, q_val, history

## 4. Сравнение

In [11]:
def run_experiments(
    problems,
    primal_solver=None,
    dual_solver=None,
    method_name="method",
    save_dir="results",
    verbose=True,
    mu: float = 10.0,
    t0: float = 1.0,
    eps: float = 0.01,
    max_iter: int = 100,
):
    save_dir = Path(save_dir)
    save_dir.mkdir(exist_ok=True)

    results = []
    for problem in problems:

        problem_id = problem["problem_id"]
        n = problem["n"]
        instance_id = problem["instance_id"]
        num_starts = len(problem["primal_starts"])
        
        for start_id in range(num_starts):
            row = {
                "method": method_name,
                "problem_id": problem_id,
                "n": n,
                "p": problem["p"],
                "instance_id": instance_id,
                "start_id": start_id,
                "mu": mu,
                "t0": t0,
                "eps": eps,
                "max_iter": max_iter,
            }

            if primal_solver is not None:
                x0 = problem["primal_starts"][start_id]
                t0_time = time.time()
                x_sol, f_val, hist_p = primal_solver(
                    problem,
                    x0,
                    mu=mu,
                    t0=t0,
                    eps=eps,
                    max_iter=max_iter
                )
                time_p = time.time() - t0_time
                final_error = hist_p['error'][-1] if hist_p['error'] else np.inf
                final_gap = hist_p['barrier_gap'][-1] if hist_p.get('barrier_gap') else np.inf
                outer_iters = len(hist_p['outer_iter'])

                total_steps = hist_p.get(
                    'total_newton_steps',
                    hist_p.get('total_bfgs_steps', 0)
                )
                row.update({
                    "primal_solution": x_sol,
                    "primal_iters": outer_iters,
                    "primal_inner_steps": total_steps,
                    "primal_time": time_p,
                    "primal_final_error": final_error,
                    "primal_final_barrier_gap": final_gap,
                    "primal_success": final_gap <= eps,
                    "primal_final_objective": f_val,
                    "primal_history": hist_p,
                })


            if dual_solver is not None:
                dual_start = problem["dual_starts"][start_id]

                t0_time = time.time()

                z_sol, q_val, hist_d = dual_solver(
                    problem,
                    dual_start,
                    mu=mu,
                    t0=t0,
                    eps=eps,
                    max_iter=max_iter
                )

                time_d = time.time() - t0_time

                p_dim = problem["p"]
                nu_sol = z_sol[:p_dim]
                lam_sol = z_sol[p_dim:]

                x_from_dual = recover_primal_from_dual(
                    nu_sol,
                    lam_sol,
                    problem["A"]
                )

                final_error = hist_d['error'][-1] if hist_d['error'] else np.inf
                final_gap = hist_d['barrier_gap'][-1] if hist_d.get('barrier_gap') else np.inf
                outer_iters = len(hist_d['outer_iter'])

                total_steps = hist_d.get(
                    'total_newton_steps',
                    hist_d.get('total_bfgs_steps', 0)
                )

                row.update({
                    "dual_solution": z_sol,
                    "dual_nu": nu_sol,
                    "dual_lambda": lam_sol,
                    "dual_recovered_x": x_from_dual,
                    "dual_iters": outer_iters,
                    "dual_inner_steps": total_steps,
                    "dual_time": time_d,
                    "dual_final_error": final_error,
                    "dual_final_barrier_gap": final_gap,
                    "dual_success": final_gap <= eps,
                    "dual_final_objective": q_val,
                    "dual_history": hist_d,
                })

            results.append(row)

            if verbose:
                msg = (
                    f"{method_name} | "
                    f"n={n} | "
                    f"inst={instance_id} | "
                    f"start={start_id} | "
                    f"μ={mu}, t0={t0}"
                )

                if primal_solver is not None:
                    msg += (
                        f" | primal_it={row['primal_iters']}"
                        f" ({row['primal_inner_steps']} inner), "
                        f"time={row['primal_time']:.4f}s, "
                        f"err={row['primal_final_error']:.2e}, "
                        f"gap={row['primal_final_barrier_gap']:.2e}"
                    )
                if dual_solver is not None:
                    msg += (
                        f" | dual_it={row['dual_iters']}"
                        f" ({row['dual_inner_steps']} inner), "
                        f"time={row['dual_time']:.4f}s, "
                        f"err={row['dual_final_error']:.2e}, "
                        f"gap={row['dual_final_barrier_gap']:.2e}"
                    )

                print(msg)


    pickle_path = save_dir / f"{method_name}_results.pkl"
    with open(pickle_path, "wb") as f:
        pickle.dump(results, f)
    print(f"\nPickle results saved to {pickle_path}")

    return results

In [12]:
newton_results = run_experiments(
    problems=problems,
    primal_solver=barrier_method_newton,
    dual_solver=barrier_method_dual_newton,
    method_name="barrier_newton",
    save_dir="results",
    verbose=True,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
)

barrier_newton | n=10 | inst=0 | start=0 | μ=10.0, t0=1.0 | primal_it=4 (24 inner), time=0.0121s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (23 inner), time=0.0161s, err=6.95e-03, gap=1.00e-02
barrier_newton | n=10 | inst=0 | start=1 | μ=10.0, t0=1.0 | primal_it=4 (24 inner), time=0.0071s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (23 inner), time=0.0090s, err=6.95e-03, gap=1.00e-02
barrier_newton | n=10 | inst=0 | start=2 | μ=10.0, t0=1.0 | primal_it=4 (23 inner), time=0.0070s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (23 inner), time=0.0091s, err=6.95e-03, gap=1.00e-02
barrier_newton | n=10 | inst=0 | start=3 | μ=10.0, t0=1.0 | primal_it=4 (23 inner), time=0.0064s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (23 inner), time=0.0085s, err=6.95e-03, gap=1.00e-02
barrier_newton | n=10 | inst=0 | start=4 | μ=10.0, t0=1.0 | primal_it=4 (24 inner), time=0.0065s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (23 inner), time=0.0083s, err=6.95e-03, gap=1.00e-02
barrier_newton | n=10 | inst=1 | start=0 | μ=10.0,

# Квази-ньютоновские методы

### BFGS для прямой барьерной задачи

Здесь важно учитывать равенство $Ax=b$. Поэтому оптимизируем не по $x$, а по координатам в ядре $A$:

$x=x_{base} + N_z$, где \
$AN=0$.

In [13]:
def bfgs_update(H, s, y):
    ys = y @ s
    if ys <= 1e-12:
        return H

    n = H.shape[0]
    I = np.eye(n)
    rho = 1.0 / ys

    return (
        (I - rho * np.outer(s, y))
        @ H
        @ (I - rho * np.outer(y, s))
        + rho * np.outer(s, s)
    )

In [17]:
def barrier_method_primal_bfgs(
    problem,
    x0,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
    inner_max_iter=100,
):
    A = problem["A"]
    b = problem["b"]
    N = problem["N"]
    f_star = problem["f_star"]
    n = problem["n"]

    x = x0.copy()
    t = t0

    history = {
        "outer_iter": [],
        "inner_iter": [],
        "objective": [],
        "error": [],
        "t_values": [],
        "total_bfgs_steps": 0,
        "step_error": [],
        "step_objective": [],
        "barrier_gap": [],
    }

    total_steps = 0
    for outer_iter in range(max_iter):
        z = np.zeros(N.shape[1])
        H = np.eye(N.shape[1])

        def phi_z(z_):
            x_ = x + N @ z_
            return barrier_function_primal(x_, t)

        def grad_phi_z(z_):
            x_ = x + N @ z_
            return N.T @ barrier_gradient_primal(x_, t)

        inner_iter = 0
        for _ in range(inner_max_iter):
            g = grad_phi_z(z)
            if np.linalg.norm(g) < 1e-8:
                break

            direction = -H @ g
            if g @ direction >= 0:
                H = np.eye(N.shape[1])
                direction = -g

            step = line_search_scalar(
                f=phi_z,
                x=z,
                p=direction,
                gamma=1.0,
                min_gamma=1e-8,
                num=30,
            )
            if step <= 1e-12:
                break

            z_new = z + step * direction
            g_new = grad_phi_z(z_new)

            s = z_new - z
            y = g_new - g

            H = bfgs_update(H, s, y)
            z = z_new

            x_current = x + N @ z
            f_current = entropy_objective(x_current)
            err_current = abs(f_current - f_star)

            history["step_objective"].append(f_current)
            history["step_error"].append(err_current)

            inner_iter += 1
            total_steps += 1

        x = x + N @ z

        f_val = entropy_objective(x)
        error = abs(f_val - f_star)

        history["outer_iter"].append(outer_iter)
        history["inner_iter"].append(inner_iter)
        history["objective"].append(f_val)
        history["error"].append(error)
        history["t_values"].append(t)
        history["total_bfgs_steps"] = total_steps
        history["barrier_gap"].append(n / t)

        if error <= eps:
            break

        t *= mu

    return x, f_val, history

In [18]:
def barrier_method_dual_bfgs(
    problem,
    nu0,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
    inner_max_iter=100,
):
    A = problem["A"]
    b = problem["b"]
    q_star = problem["q_star"]

    p = problem["p"]
    n = problem["n"]

    nu = nu0.copy()
    lam = np.ones(n)

    z = np.concatenate([nu, lam])
    t = t0

    history = {
        "outer_iter": [],
        "inner_iter": [],
        "objective": [],
        "error": [],
        "t_values": [],
        "total_bfgs_steps": 0,
        "step_error": [],
        "step_objective": [],
        "barrier_gap": [],
    }

    total_steps = 0

    for outer_iter in range(max_iter):
        H = np.eye(p + n)
        inner_iter = 0

        for _ in range(inner_max_iter):
            g = dual_phi_grad(z, A, b, t)
            if np.linalg.norm(g) < 1e-8:
                break

            direction = -H @ g
            if g @ direction >= 0:
                H = np.eye(p + n)
                direction = -g

            step = line_search_scalar(
                f=lambda z_: dual_phi(z_, A, b, t),
                x=z,
                p=direction,
                gamma=1.0,
                min_gamma=1e-8,
                num=30,
            )

            if step <= 1e-12:
                break

            z_new = z + step * direction
            if np.any(z_new[p:] <= 0):
                break

            g_new = dual_phi_grad(z_new, A, b, t)
            s = z_new - z
            y = g_new - g

            H = bfgs_update(H, s, y)
            z = z_new

            nu_current = z[:p]
            lam_current = z[p:]

            exp_term = np.exp(-1 - A.T @ nu_current + lam_current)
            phi_val = (
                b @ nu_current
                - np.sum(lam_current)
                + np.sum(exp_term)
            )

            q_val = -phi_val
            err_current = abs(q_val - q_star)

            history["step_objective"].append(q_val)
            history["step_error"].append(err_current)

            inner_iter += 1
            total_steps += 1

        nu = z[:p]
        lam = z[p:]

        exp_term = np.exp(-1 - A.T @ nu + lam)
        phi_val = (
            b @ nu
            - np.sum(lam)
            + np.sum(exp_term)
        )

        q_val = -phi_val
        error = abs(q_val - q_star)

        history["outer_iter"].append(outer_iter)
        history["inner_iter"].append(inner_iter)
        history["objective"].append(q_val)
        history["error"].append(error)
        history["t_values"].append(t)
        history["total_bfgs_steps"] = total_steps
        history["barrier_gap"].append(n / t)

        if error <= eps:
            break

        t *= mu

    return z, q_val, history

In [19]:
bfgs_results = run_experiments(
    problems=problems,
    primal_solver=barrier_method_primal_bfgs,
    dual_solver=barrier_method_dual_bfgs,
    method_name="barrier_bfgs",
    save_dir="results",
    verbose=True,
    mu=10.0,
    t0=1.0,
    eps=0.01,
    max_iter=100,
)

barrier_bfgs | n=10 | inst=0 | start=0 | μ=10.0, t0=1.0 | primal_it=4 (63 inner), time=0.0204s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (121 inner), time=0.0452s, err=6.95e-03, gap=1.00e-02
barrier_bfgs | n=10 | inst=0 | start=1 | μ=10.0, t0=1.0 | primal_it=4 (64 inner), time=0.0181s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (122 inner), time=0.0385s, err=6.95e-03, gap=1.00e-02
barrier_bfgs | n=10 | inst=0 | start=2 | μ=10.0, t0=1.0 | primal_it=4 (64 inner), time=0.0173s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (123 inner), time=0.0393s, err=6.95e-03, gap=1.00e-02
barrier_bfgs | n=10 | inst=0 | start=3 | μ=10.0, t0=1.0 | primal_it=4 (64 inner), time=0.0184s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (122 inner), time=0.0408s, err=6.95e-03, gap=1.00e-02
barrier_bfgs | n=10 | inst=0 | start=4 | μ=10.0, t0=1.0 | primal_it=4 (63 inner), time=0.0172s, err=3.05e-03, gap=1.00e-02 | dual_it=4 (120 inner), time=0.0429s, err=6.95e-03, gap=1.00e-02
barrier_bfgs | n=10 | inst=1 | start=0 | μ=10.0, t0=1.0